In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mis_dro.likelihood import cholesky_param_to_covariance
from mis_dro.bayes_conjugates import *
from mis_dro.dataset import portfolio_dataset
from mis_dro.preprocessing import normalise_by_dimension

In [ ]:
dgp = "DowJones"
replication = 0
normalise = False
npl_uuid = "83aa670b-a7d2-4c4c-8472-3656114be630"
samples_dir = Path("/dcs/large/u1508153/misdro/mmd_portfolio_debug/2025_01_15_mmd_portfolio")

mmc2_dir = Path("/dcs/pg20/u1508153/datasets/misdro/mmc2")
assert mmc2_dir.exists()
# returns_df = pd.read_excel(mmc2_dir / "Datasets" / dgp / f"{dgp}.xlsx", sheet_name="Assets_Returns", header=None)
data, data_eval = portfolio_dataset(dgp, replication, mmc2_dir)

npl_settings_df = pd.read_csv(samples_dir / "npl_settings.csv").set_index("npl_uuid")

if normalise:
    data = normalise_by_dimension(data)

portfolio_npl_samples_csv = Path(samples_dir / npl_uuid / f"npl_sample_{replication}.csv")
# portfolio_npl_samples_csv = Path(f"/dcs/large/u1508153/misdro/mmd_portfolio_samples/8c01253f-8d7b-471c-a7b3-3516b2ddef2e/npl_sample_{replication}.csv")

npl_df = pd.read_csv(portfolio_npl_samples_csv, index_col=False, header=None)
npl_samples = npl_df.values

num_posterior_samples = npl_samples.shape[0]
dim = data.shape[1]

settings = npl_settings_df.loc[npl_uuid]
settings

In [ ]:
# Bayes posterior
posterior = "normal_inverse_wishart"
theta_prior = default_prior_params(posterior, dim=dim)
theta_posterior = get_posterior_params(posterior, data, theta_prior)
mu_post, _, iota_post, Psi_post = theta_posterior

In [ ]:
# NPL posterior with full covariance
npl_means = np.zeros((num_posterior_samples, dim))
npl_covs = np.zeros((num_posterior_samples, dim, dim))
for i in range(num_posterior_samples):
    npl_means[i] = npl_samples[i,:dim]
    npl_covs[i] = cholesky_param_to_covariance(dim, npl_samples[i,dim:])
mu_post

In [ ]:
mean_npl_mean = npl_means.mean(axis=0)
mean_npl_cov = npl_covs.mean(axis=0)
mean_npl_mean

In [ ]:
mean_npl_mean - mu_post

In [ ]:
# Create a figure and axes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

bayes_cov = (1.0 / float(iota_post)) * Psi_post
vmin = np.min([bayes_cov, mean_npl_cov])
vmax = np.max([bayes_cov, mean_npl_cov])

# Plot the first matrix
c1 = ax1.matshow(mean_npl_cov, cmap='viridis', vmin=vmin, vmax=vmax)
ax1.set_title(f"Mean of NPL cov samples (eta={settings['eta']})")

# Plot the second matrix
c2 = ax2.matshow(bayes_cov, cmap='viridis', vmin=vmin, vmax=vmax)
ax2.set_title('Expected Bayes cov')

# Create a shared colorbar
cbar = fig.colorbar(c1, ax=[ax1, ax2], orientation='vertical')

fig.show()


In [ ]:
(data[:,0] - data[:,0].mean())/data[:,0].std()